# Day 8

In [1]:
import fs from 'node:fs';

In [2]:
const input = fs.readFileSync("input.txt", "utf-8");

In [3]:
const sample = `\
162,817,812
57,618,57
906,360,560
592,479,940
352,342,300
466,668,158
542,29,236
431,825,988
739,650,466
52,470,668
216,146,977
819,987,18
117,168,530
805,96,715
346,949,466
970,615,88
941,993,340
862,61,35
984,92,344
425,690,689`

## Part 1

The input consists of rows of 3d coordinates (x,y,z). We can think of each point as a node in a graph, and between every two points there's an edge weighted by the distance. We are then asked to keep the smallest 1000 edges, and in the resulting graph multiply the sizes of the three largest connected components. Funnily, this is kinda similar to stuff I did in my Master's. Anyway, this is something I wish I could use numpy for, but since we chose javascript, no way around it...

Let's start by processing the input and calculating the distance matrix.

In [4]:
const process = input => input.split("\n").map(s => s.split(",").map(n => parseInt(n)));
const pts = process(sample);

And a helper for creating a zero matrix which we will populate

In [5]:
const zeroes = n => Array.from({length: n}, (_, i) => Array.from({length: n}, (_, i) => 0))
zeroes(5);

[
  [ 0, 0, 0, 0, 0 ],
  [ 0, 0, 0, 0, 0 ],
  [ 0, 0, 0, 0, 0 ],
  [ 0, 0, 0, 0, 0 ],
  [ 0, 0, 0, 0, 0 ]
]

and a helper for calculating euclidean .

I would like to use something like `zip()` in python but seems like it doesn't exist. Either we use a loop or implement iterators or unroll the loop.

In [6]:
const euc = (a, b) => [a[0] - b[0], a[1] - b[1], a[2] - b[2]].map(x => x*x).reduce((acc, x) => acc + x) ** 0.5;
euc([1,0,0], [0,0,1]);

1.4142135623730951

In [7]:
function dists(pts) {
  const n = pts.length;
  const D = zeroes(n);
  for (let i = 0; i < n; i++) {
    for (let j = 0; j < n; j++) {
      D[i][j] = euc(pts[i], pts[j]);
    }
  }
  return D;
}

dists(pts.slice(0,3));

[
  [ 0, 787.814064357828, 908.7843528582565 ],
  [ 787.814064357828, 0, 1019.9872548223335 ],
  [ 908.7843528582565, 1019.9872548223335, 0 ]
]

Okay, looks solid.

In [8]:
const D = dists(pts);

You know what, I think in this case it would be easier to work with a flattened list.

In [33]:
const D_list = [];
for (let i = 0; i < D.length; i++) {
  for (let j = i+1; j < D[i].length; j++) { // from symmetry we only need the upper triangular part
    D_list.push([D[i][j], [i, j]]);
  }
}
D_list.slice(0,10);

[
  [ 787.814064357828, [ 0, 1 ] ],
  [ 908.7843528582565, [ 0, 2 ] ],
  [ 561.7187908553532, [ 0, 3 ] ],
  [ 723.7879523727927, [ 0, 4 ] ],
  [ 736.432617419951, [ 0, 5 ] ],
  [ 1047.4349621814235, [ 0, 6 ] ],
  [ 321.560258738545, [ 0, 7 ] ],
  [ 693.2055972076394, [ 0, 8 ] ],
  [ 391.46519640959144, [ 0, 9 ] ],
  [ 693.0959529531247, [ 0, 10 ] ]
]

Now lets sort and take the first 10.

In [34]:
const shortest = D_list.sort((a, b) => a[0] - b[0]).slice(0, 10);

Let's check that we're getting the same edges like in the problem. Should have

- 162,817,812 and 425,690,689
- 162,817,812 and 431,825,988
- 906,360,560 and 805,96,715
- 431,825,988 and 425,690,689

In [35]:
shortest.map(e => [pts[e[1][0]], pts[e[1][1]]]).slice(0, 4)

[
  [ [ 162, 817, 812 ], [ 425, 690, 689 ] ],
  [ [ 162, 817, 812 ], [ 431, 825, 988 ] ],
  [ [ 906, 360, 560 ], [ 805, 96, 715 ] ],
  [ [ 431, 825, 988 ], [ 425, 690, 689 ] ]
]

Looks correct. Let's put it in a function

In [48]:
function shortest_edges(pts, n_edges) {
  const D = dists(pts);
  
  const D_list = [];
  for (let i = 0; i < D.length; i++) {
    for (let j = i+1; j < D[i].length; j++) { // from symmetry we only need the upper triangular part
      D_list.push([D[i][j], [i, j]]);
    }
  }
  
  const shortest = D_list.sort((a, b) => a[0] - b[0]).slice(0, n_edges).map(e => e[1]);
  return shortest;
}
shortest_edges(pts)

[
  [ 0, 19 ],  [ 0, 7 ],   [ 2, 13 ],  [ 7, 19 ],  [ 17, 18 ], [ 9, 12 ],
  [ 11, 16 ], [ 2, 8 ],   [ 14, 19 ], [ 2, 18 ],  [ 3, 19 ],  [ 4, 6 ],
  [ 4, 12 ],  [ 4, 5 ],   [ 6, 17 ],  [ 3, 7 ],   [ 8, 19 ],  [ 0, 9 ],
  [ 11, 15 ], [ 13, 18 ], [ 5, 8 ],   [ 0, 14 ],  [ 8, 16 ],  [ 1, 5 ],
  [ 9, 19 ],  [ 5, 14 ],  [ 8, 15 ],  [ 15, 16 ], [ 10, 12 ], [ 6, 18 ],
  [ 1, 4 ],   [ 9, 10 ],  [ 4, 9 ],   [ 3, 13 ],  [ 8, 14 ],  [ 5, 11 ],
  [ 3, 10 ],  [ 2, 3 ],   [ 5, 15 ],  [ 4, 8 ],   [ 3, 8 ],   [ 4, 19 ],
  [ 5, 19 ],  [ 6, 12 ],  [ 2, 15 ],  [ 7, 14 ],  [ 6, 13 ],  [ 0, 3 ],
  [ 8, 11 ],  [ 15, 17 ], [ 15, 18 ], [ 2, 6 ],   [ 9, 14 ],  [ 2, 19 ],
  [ 1, 14 ],  [ 5, 16 ],  [ 3, 9 ],   [ 2, 17 ],  [ 14, 16 ], [ 7, 9 ],
  [ 8, 13 ],  [ 2, 4 ],   [ 8, 18 ],  [ 12, 19 ], [ 1, 9 ],   [ 4, 14 ],
  [ 7, 8 ],   [ 4, 17 ],  [ 10, 13 ], [ 5, 6 ],   [ 10, 19 ], [ 11, 14 ],
  [ 1, 12 ],  [ 4, 13 ],  [ 2, 5 ],   [ 2, 16 ],  [ 4, 18 ],  [ 13, 17 ],
  [ 5, 9 ],   [ 6, 8 ],   [ 16, 19 ], [ 0, 10 ],  [ 

The final step is to now find the connected components in the resulting graph. As far as I remember this is done using BFS. We will need to build a neighbors map that maps and index to all neighbors.

In [49]:
const neighbors = new Map();
for (let i = 0; i < pts.length; i++) neighbors.set(i, new Set());
neighbors

Map(20) {
  0 => Set(0) {},
  1 => Set(0) {},
  2 => Set(0) {},
  3 => Set(0) {},
  4 => Set(0) {},
  5 => Set(0) {},
  6 => Set(0) {},
  7 => Set(0) {},
  8 => Set(0) {},
  9 => Set(0) {},
  10 => Set(0) {},
  11 => Set(0) {},
  12 => Set(0) {},
  13 => Set(0) {},
  14 => Set(0) {},
  15 => Set(0) {},
  16 => Set(0) {},
  17 => Set(0) {},
  18 => Set(0) {},
  19 => Set(0) {}
}

In [50]:
shortest.forEach(([d, [i, j]]) => {
  neighbors.get(i).add(j);
  neighbors.get(j).add(i);
});
neighbors

Map(20) {
  0 => Set(2) { 19, 7 },
  1 => Set(0) {},
  2 => Set(3) { 13, 8, 18 },
  3 => Set(0) {},
  4 => Set(0) {},
  5 => Set(0) {},
  6 => Set(0) {},
  7 => Set(2) { 0, 19 },
  8 => Set(1) { 2 },
  9 => Set(1) { 12 },
  10 => Set(0) {},
  11 => Set(1) { 16 },
  12 => Set(1) { 9 },
  13 => Set(1) { 2 },
  14 => Set(1) { 19 },
  15 => Set(0) {},
  16 => Set(1) { 11 },
  17 => Set(1) { 18 },
  18 => Set(2) { 17, 2 },
  19 => Set(3) { 0, 7, 14 }
}

In [51]:
function get_neighbors_map(pts, edges) {
  const neighbors = new Map();
  for (let i = 0; i < pts.length; i++) neighbors.set(i, new Set());
  
  edges.forEach(([i, j]) => {
    neighbors.get(i).add(j);
    neighbors.get(j).add(i);
  });
  return neighbors;
}

In [52]:
const added = new Set([0]);
const queue = [0];
let size = 0;
while (queue.length > 0) {
  const cur = queue.shift(); // simulate queue, but O(n)...
  console.log(cur);
  size += 1;
  neighbors.get(cur).forEach(j => {
    if (added.has(j)) return;
    queue.push(j);
    added.add(j);
  });
}
size

0
19
7
14


4

Now we need to figure out how given a node we find the size of the connected component.

Okay, this seems to work. Lets put it in a function that gets a starting node and neighbors map and returns the CC

In [53]:
function CC(node, neighbors) {
  const added = new Set([node]);
  const queue = [node];
  while (queue.length > 0) {
    const cur = queue.shift(); // simulate queue, but O(n)...
    neighbors.get(cur).forEach(j => {
      if (added.has(j)) return;
      queue.push(j);
      added.add(j);
    });
  }
  return added;
}
CC(0, neighbors)

Set(4) { 0, 19, 7, 14 }

Now we need an outer loop

In [54]:
let visited = new Set();
const cc_sizes = [];
for (let i=0; i < pts.length; i++) {
  if (visited.has(i)) continue;
  const cc = CC(i, neighbors);
  cc_sizes.push(cc.size);
  visited = visited.union(cc);
}
[visited.size, cc_sizes]

[
  20,
  [
    4, 1, 5, 1, 1,
    1, 1, 2, 1, 2,
    1
  ]
]

In [55]:
cc_sizes.sort((a, b) => b - a).slice(0,3).reduce((acc, x) => acc * x);

40

Let's wrap everything up

In [63]:
function part1(input, n_edges) {
  const pts = process(input);
  const edges = shortest_edges(pts, n_edges);
  // console.log(edges)
  const neighbors = get_neighbors_map(pts, edges);
  let visited = new Set();
  const cc_sizes = [];
  for (let i=0; i < pts.length; i++) {
    if (visited.has(i)) continue;
    const cc = CC(i, neighbors);
    cc_sizes.push(cc.size);
    visited = visited.union(cc);
  }
  return cc_sizes.sort((a, b) => b - a).slice(0,3).reduce((acc, x) => acc * x);
}
part1(sample, 10);

40

In [68]:
process(input).length

1000

Correcto!

Now we need to keep adding edges until the graph is fully connected, and then multiply the x coordinates of the last edge. We can think of a smart algorithm, or we can do some binary search. If the result of the previous part is 1000 then the graph is almost surely connected, so we can just look for lowest number of edges needed.

In [67]:
part1(input, 100000);

1000

In [69]:
part1(input, 10000);

1000

In [70]:
part1(input, 1000);

127551

In [71]:
part1(input, 5000);

1000

In [72]:
part1(input, 2500);

60543

In [73]:
part1(input, 3750);

999

In [74]:
part1(input, 3800);

1000

In [75]:
part1(input, 3770);

999

In [76]:
part1(input, 3785);

1000

In [77]:
part1(input, 3778);

1000

In [78]:
part1(input, 3774);

999

In [79]:
part1(input, 3776);

999

In [80]:
part1(input, 3777);

999

so after 3778 edges the graph is connected. Which edge is that then?

In [83]:
const pts = process(input);
const D = dists(pts);

const D_list = [];
for (let i = 0; i < D.length; i++) {
  for (let j = i+1; j < D[i].length; j++) { // from symmetry we only need the upper triangular part
    D_list.push([D[i][j], [i, j]]);
  }
}

const shortest = D_list.sort((a, b) => a[0] - b[0]);
shortest[3777];

[ 12925.06947756955, [ 180, 557 ] ]

In [84]:
[pts[180], pts[557]]

[ [ 44966, 19938, 98403 ], [ 52200, 30637, 98911 ] ]

In [85]:
44966 * 52200

2347225200

Noice! in AOC, sometimes you are the algoirthm.

## Smarter approaches

Looked at some solutions online. So for part 2, the question is equivalent to taking the largest edge in the minimal spanning tree. A bit sad that I didn't think of that, but it's okay, I don't deal too much with graphs day to day. Anyway, Kruskal's algorithm apparently solves that, so we can try and implement it.